##### ***文本预处理***
###### 为处理序列数据，我们不仅需要统计工具，而且需要对序列数据进行预处理。其中文本是最常见的例子之一。文本预处理的常见预处理步骤如下：<br>1. 将文本作为字符串加载到内存中<br>2. 将字符串拆分为词元序列（token sequence）<br>3. 建立一个词表，将拆分的词元映射到数字索引。<br>4. 将文本转化为数字索引，输入到模型中。

In [1]:
import collections
import re 
from d2l import torch as d2l

In [2]:
# 1.读取数据集
#@save
d2l.DATA_HUB['time_machine'] = (d2l.DATA_URL + 'timemachine.txt',
                                '090b5e7e70c295757f55df93cb0a180b9691891a')

def read_time_machine(): #@save
    """将时间机器数据集加载到文本行的列表中"""
    with open(d2l.download('time_machine'), 'r') as f:
        lines = f.readlines()
    return [re.sub('[^A-Za-z]+', ' ', line).strip().lower() for line in lines] # 利用正则表达式将非字母字符替换为空空格，再将首尾空格去掉，最后转换为小写

lines = read_time_machine()
print(f'#文本总行数：{len(lines)}')
print(lines[0])
print(lines[10])


#文本总行数：3221
the time machine by h g wells
twinkled and his usually pale face was flushed and animated the


###### 利用tokenize函数将文本行列表作为输入，列表中的每一个元素都是文本序列，每个文本序列又被拆分成一个词元列表，其中每个词元都是一个字符串，最终返回一个由词元列表组成的列表。

In [3]:
# 2.词元化
def tokenize(lines, token='word'): # 词元粒度，word(默认)或char
    """将文本行拆分为词元序列"""
    if token == 'word':
        return [line.split() for line in lines] # 按词切分
    elif token == 'char':
        return [list(line) for line in lines] # 按字符切分
    else:
        raise ValueError(f'Unknown token type: {token}')

tokens = tokenize(lines)
for i in range(11):
    print(tokens[i])

['the', 'time', 'machine', 'by', 'h', 'g', 'wells']
[]
[]
[]
[]
['i']
[]
[]
['the', 'time', 'traveller', 'for', 'so', 'it', 'will', 'be', 'convenient', 'to', 'speak', 'of', 'him']
['was', 'expounding', 'a', 'recondite', 'matter', 'to', 'us', 'his', 'grey', 'eyes', 'shone', 'and']
['twinkled', 'and', 'his', 'usually', 'pale', 'face', 'was', 'flushed', 'and', 'animated', 'the']


###### 将文本序列转换为词元序列后，我们需要将词元序列转化为数字索引，即构建词元表。<br>首先对词元序列进行统计，得到唯一词元，将其成为语料（corpus）。然后根据词元出现的频率，为其分配一个数字索引，通常移除那些出现频率低的词元，这样可以降低复杂度。另外，对于语料库中不存在或者已被删除的任何词元都将映射到一个特定的未知词元（“\<unk>”）。我们可以选择增加一个列表，用于保存那些被保留的词元，也就是说除了真实词，还可以加一些特殊词元来帮组模型来理解， 例如：填充词元（“\<pad>”，让不同长度的序列对齐）； 序列开始词元（“\<bos>”，标记序列开始）； 序列结束词元（“\<eos>”，标记序列结束）。

In [4]:
class Vocab: #@save
    """词元表"""
    def __init__(self, tokens=None, min_freq=0, reserved_tokens=None):
        if tokens is None:
            tokens = []
        if reserved_tokens is None: # 保留词元，如填充词元（“\<pad>”）； 序列开始词元（“\<bos>”）； 序列结束词元（“\<eos>”）
            reserved_tokens = []

        # 按词元出现频率排序
        counter = count_corpus(tokens)
        self._token_freqs = sorted(counter.items(), key=lambda x: x[1], reverse=True) # 按词元出现频率排序，从高到低

        # 未知词元的索引为0
        self.idx_to_token = ['<unk>'] + reserved_tokens # 构建词元列表，将保留词元放开头
        self.token_to_idx = {token: idx for idx, token in enumerate(self.idx_to_token)} # 构建词元索引字典，将词元映射到索引，索引从0开始，未知词元的索引为0

        for token, freq in self._token_freqs:
            if freq < min_freq: # 如果词元出现频率小于最小频率，说明词元出现次数太少，不考虑，后面的词元也不考虑因为是按频率排序的
                break
            if token not in self.token_to_idx:
                self.idx_to_token.append(token)
                self.token_to_idx[token] = len(self.idx_to_token) - 1

    def __len__(self): # 返回词元表的大小
        return len(self.idx_to_token)
    
    def __getitem__(self, tokens): # 词元->索引
        if not isinstance(tokens, (list, tuple)): # 如果不在
            return self.token_to_idx.get(tokens, self.unk)
        return [self.__getitem__(token) for token in tokens]
    
    def to_tokens(self, indices): # 索引->词元
        if not isinstance(indices, (list, tuple)):
            return self.idx_to_token[indices]
        return [self.idx_to_token[index] for index in indices]

    @property
    def unk(self): # 未知词元的索引为0
        return 0

    @property # 用于获取属性的值，而不需要调用方法，只读保护
    def token_freqs(self):
        return self._token_freqs

def count_corpus(tokens): #@save
    """统计词元出现频率"""
    # 如果tokens为空或tokens的第一个元素是一个列表，说明tokens是一个二维列表，需要先展平为一维列表
    if len(tokens) == 0 or isinstance(tokens[0], list): # isinstance用来判断一个对象是否是一个已知类型
        # 将tokens列表展平
        tokens = [token for line in tokens for token in line]
    return collections.Counter(tokens)

In [5]:
vocab = Vocab(tokens)
print(list(vocab.token_to_idx.items())[:10])

[('<unk>', 0), ('the', 1), ('i', 2), ('and', 3), ('of', 4), ('a', 5), ('to', 6), ('was', 7), ('in', 8), ('that', 9)]


In [6]:
for i in [0, 10]:
    print("文本", tokens[i])
    print("索引", vocab[tokens[i]])

文本 ['the', 'time', 'machine', 'by', 'h', 'g', 'wells']
索引 [1, 19, 50, 40, 2183, 2184, 400]
文本 ['twinkled', 'and', 'his', 'usually', 'pale', 'face', 'was', 'flushed', 'and', 'animated', 'the']
索引 [2186, 3, 25, 1044, 362, 113, 7, 1421, 3, 1045, 1]


In [ ]:
def load_corpus_time_machine(max_tokens=-1): #@save
    """返回时光机器数据集的词元索引列表和词表"""
    lines = read_time_machine()
    tokens = tokenize(lines, 'char')
    vocab = Vocab(tokens)

    # 因为数据集中的每个文本行不一定是一个句子或者段落，所以将所有文本行平展到一个列表中
    corpus = [vocab[token] for line in tokens for token in line]
    if max_tokens > 0:
        corpus = corpus[:max_tokens]
    return corpus, vocab
corpus, vocab = load_corpus_time_machine()
len(corpus), len(vocab)

(170580, 28)

: 